# BankruptcySense AI — Bankruptcy Prediction for Small Businesses

**Model:** Random Forest + SMOTE + RandomizedSearchCV  
**Dataset:** Polish Bankruptcy Dataset (UCI) — 5year.arff (~5,910 rows, 64 features)  
**Target:** Binary — 0 = Safe, 1 = Bankrupt  

---
### Pipeline
1. Install dependencies
2. Upload dataset
3. Load & explore
4. Drop high-missing columns
5. Stratified train/test split ← first
6. Impute → clip → scale (fit on train only)
7. SMOTE (train only)
8. Feature selection (top 30)
9. RandomizedSearchCV tuning
10. OOF threshold tuning
11. Evaluation + plots
12. Download model artifacts

## 1 · Install Dependencies

In [ ]:
!pip install imbalanced-learn==0.14.1 -q
print('✓ imbalanced-learn installed')

## 2 · Upload Dataset

In [ ]:
from google.colab import files
print('Upload 5year.arff:')
uploaded = files.upload()
ARFF_PATH = list(uploaded.keys())[0]
print(f'✓ Uploaded: {ARFF_PATH}')

## 3 · Imports & Constants

In [ ]:
import pickle, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy.io import arff
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, recall_score, roc_auc_score, classification_report, confusion_matrix, precision_recall_curve
from imblearn.over_sampling import SMOTE
warnings.filterwarnings('ignore')

# ── Constants ──────────────────────────────────────────────────
TOP_N_FEATURES       = 30
TEST_SIZE            = 0.2
RANDOM_STATE         = 42
MISSING_DROP_THRESH  = 0.40
OUTLIER_LOW          = 1
OUTLIER_HIGH         = 99
N_ITER               = 30
CV_FOLDS             = 5
MIN_RECALL_FLOOR     = 0.75
MIN_PRECISION_FLOOR  = 0.20

RF_PARAM_DIST = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', 0.3],
    'bootstrap':         [True],
    'class_weight':      ['balanced', 'balanced_subsample'],
}
print('✓ Imports done')

## 4 · Load & Explore Data

In [ ]:
raw, meta = arff.loadarff(ARFF_PATH)
df = pd.DataFrame(raw)
for col in df.select_dtypes([object]).columns:
    df[col] = df[col].str.decode('utf-8')
if 'class' in df.columns:
    df.rename(columns={'class': 'target'}, inplace=True)
df['target'] = pd.to_numeric(df['target'], errors='coerce').astype(int)

print(f'Shape: {df.shape}')
print(f'Bankrupt rate: {df["target"].mean():.2%}')
print(f'Class counts:\n{df["target"].value_counts()}')
df.head(3)

In [ ]:
# Missing value heatmap
missing = df.isnull().mean().sort_values(ascending=False)
print(f'Columns with >40% missing: {list(missing[missing > 0.4].index)}')
plt.figure(figsize=(14,3))
plt.bar(range(len(missing)), missing.values, color='steelblue')
plt.axhline(0.4, color='red', linestyle='--', label='40% threshold')
plt.xlabel('Feature index'); plt.ylabel('Missing fraction')
plt.title('Missing Value Rate per Feature'); plt.legend(); plt.tight_layout(); plt.show()

## 5 · Drop High-Missing Columns

In [ ]:
missing_frac = df.isnull().mean()
cols_to_drop = missing_frac[missing_frac > MISSING_DROP_THRESH].index.tolist()
print(f'Dropping {len(cols_to_drop)} columns: {cols_to_drop}')
df = df.drop(columns=cols_to_drop)
print(f'Shape after drop: {df.shape}')

## 6 · Stratified Train/Test Split  ← First, before any fitting

In [ ]:
X = df.drop(columns=['target'])
y = df['target']
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train bankrupt rate: {y_train.mean():.2%}  |  Test: {y_test.mean():.2%}')

## 7 · Impute → Clip → Scale  (fit on train only)

In [ ]:
# Median imputation
imputer = SimpleImputer(strategy='median')
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=feature_names)
X_test  = pd.DataFrame(imputer.transform(X_test),      columns=feature_names)

# Outlier clipping (percentiles from train)
lower = X_train.quantile(OUTLIER_LOW  / 100)
upper = X_train.quantile(OUTLIER_HIGH / 100)
X_train = X_train.clip(lower=lower, upper=upper, axis=1)
X_test  = X_test.clip( lower=lower, upper=upper, axis=1)

# StandardScaler
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print('✓ Impute → Clip → Scale done')

## 8 · SMOTE Oversampling  (train only)

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train_sc, y_train)
unique, counts = np.unique(y_train_res, return_counts=True)
print(f'After SMOTE: {dict(zip(unique, counts))}')

# Class distribution plot
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for ax, (title, vals) in zip(axes, [('Before SMOTE', y_train), ('After SMOTE', y_train_res)]):
    u, c = np.unique(vals, return_counts=True)
    ax.bar(['Safe','Bankrupt'], c, color=['steelblue','tomato'])
    ax.set_title(title); ax.set_ylabel('Count')
plt.tight_layout(); plt.show()

## 9 · Feature Selection  (top 30 by RF importance)

In [ ]:
selector = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                   n_jobs=-1, class_weight='balanced')
selector.fit(X_train_res, y_train_res)

importances  = selector.feature_importances_
top_indices  = np.argsort(importances)[::-1][:TOP_N_FEATURES]
top_features = [feature_names[i] for i in top_indices]

X_train_sel = X_train_res[:, top_indices]
X_test_sel  = X_test_sc[:,  top_indices]

print(f'Selected features: {top_features}')

# Feature importance bar chart
plt.figure(figsize=(10, 5))
plt.bar(range(TOP_N_FEATURES), importances[top_indices], color='steelblue')
plt.xticks(range(TOP_N_FEATURES), top_features, rotation=45, ha='right', fontsize=8)
plt.title(f'Top {TOP_N_FEATURES} Feature Importances'); plt.tight_layout(); plt.show()

## 10 · Hyperparameter Tuning  (RandomizedSearchCV + StratifiedKFold)

In [ ]:
cv     = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=RF_PARAM_DIST,
    n_iter=N_ITER, cv=cv, scoring='recall',
    refit=True, verbose=1, random_state=RANDOM_STATE, n_jobs=-1
)
search.fit(X_train_sel, y_train_res)
best_model = search.best_estimator_

print(f'\nBest params : {search.best_params_}')
print(f'Best CV recall: {search.best_score_:.4f}')

## 11 · Cross-Validation Stability Check

In [ ]:
scores = cross_val_score(best_model, X_train_sel, y_train_res,
                         cv=cv, scoring='recall', n_jobs=-1)
print(f'CV recall scores : {np.round(scores, 4)}')
print(f'Mean: {scores.mean():.4f}  |  Std: {scores.std():.4f}')
print('✓ Stable' if scores.std() < 0.05 else '✗ High variance — check data')

## 12 · OOF Threshold Tuning  (maximise F1 s.t. recall ≥ 0.75)

In [ ]:
# Get feature-selected version of original (pre-SMOTE) train
X_train_orig_sel = X_train_sc[:, top_indices]

oof_proba = np.zeros(len(y_train))
cv_oof    = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

for tr_idx, val_idx in cv_oof.split(X_train_orig_sel, y_train):
    X_tr, X_val = X_train_orig_sel[tr_idx], X_train_orig_sel[val_idx]
    y_tr        = np.array(y_train)[tr_idx]
    X_tr_res, y_tr_res = SMOTE(random_state=RANDOM_STATE).fit_resample(X_tr, y_tr)
    params = {k: v for k, v in best_model.get_params().items()
              if k not in ('random_state', 'n_estimators', 'n_jobs')}
    fold_m = RandomForestClassifier(**params, n_estimators=100,
                                    n_jobs=-1, random_state=RANDOM_STATE)
    fold_m.fit(X_tr_res, y_tr_res)
    oof_proba[val_idx] = fold_m.predict_proba(X_val)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_train, oof_proba)
f1s = np.where((precisions[:-1]+recalls[:-1])>0,
               2*precisions[:-1]*recalls[:-1]/(precisions[:-1]+recalls[:-1]), 0)

valid = (precisions[:-1] >= MIN_PRECISION_FLOOR) & (recalls[:-1] >= MIN_RECALL_FLOOR)
masked = np.where(valid, f1s, 0)
best_idx = np.argmax(masked) if masked.max() > 0 else np.argmax(f1s)
THRESHOLD = float(thresholds[best_idx])

print(f'OOF precision @ threshold : {precisions[best_idx]:.4f}')
print(f'OOF recall    @ threshold : {recalls[best_idx]:.4f}')
print(f'OOF F1        @ threshold : {f1s[best_idx]:.4f}')
print(f'Optimal threshold         : {THRESHOLD:.4f}')

## 13 · Final Evaluation on Test Set

In [ ]:
y_proba = best_model.predict_proba(X_test_sel)[:, 1]
y_pred  = (y_proba >= THRESHOLD).astype(int)

f1      = f1_score(y_test, y_pred)
recall  = recall_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)
cm      = confusion_matrix(y_test, y_pred)

print('='*55)
print('  TEST-SET RESULTS')
print('='*55)
print(f'  Threshold  : {THRESHOLD:.4f}')
print(f'  ROC-AUC    : {roc_auc:.4f}  {"✓" if roc_auc > 0.90 else "✗ target > 0.90"}')
print(f'  Recall     : {recall:.4f}   {"✓" if recall  > 0.75 else "✗ target > 0.75"}')
print(f'  F1 Score   : {f1:.4f}   {"✓" if f1      > 0.72 else "✗ target > 0.72"}')
print('='*55)
print(f'\n{classification_report(y_test, y_pred, target_names=["Safe","Bankrupt"])}')

In [ ]:
from sklearn.metrics import roc_curve, auc as sk_auc

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('BankruptcySense AI — Model Evaluation', fontsize=13, fontweight='bold')

# Confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Safe','Bankrupt'], yticklabels=['Safe','Bankrupt'], ax=axes[0])
axes[0].set_title('Confusion Matrix'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {sk_auc(fpr,tpr):.3f}')
axes[1].plot([0,1],[0,1],'navy',lw=1,linestyle='--')
axes[1].set_title('ROC Curve'); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].legend(loc='lower right')

# Feature importances
imp = best_model.feature_importances_
sidx = np.argsort(imp)
axes[2].barh([top_features[i] for i in sidx], imp[sidx], color='steelblue')
axes[2].set_title(f'Top {TOP_N_FEATURES} Feature Importances'); axes[2].set_xlabel('Importance')

plt.tight_layout(); plt.savefig('evaluation.png', dpi=150, bbox_inches='tight'); plt.show()
print('✓ Saved evaluation.png')

## 14 · Save & Download Model Artifacts

In [ ]:
import os, zipfile
os.makedirs('model', exist_ok=True)

with open('model/rf_model.pkl',   'wb') as f: pickle.dump(best_model,   f)
with open('model/scaler.pkl',     'wb') as f: pickle.dump(scaler,       f)
with open('model/features.pkl',   'wb') as f: pickle.dump(top_features, f)
with open('model/threshold.pkl',  'wb') as f: pickle.dump(THRESHOLD,    f)

# Zip all artifacts for easy download
with zipfile.ZipFile('model_artifacts.zip', 'w') as z:
    for fname in ['rf_model.pkl','scaler.pkl','features.pkl','threshold.pkl']:
        z.write(f'model/{fname}', fname)
    z.write('evaluation.png', 'evaluation.png')

print('Saved:')
for f in ['rf_model.pkl','scaler.pkl','features.pkl','threshold.pkl']:
    size = os.path.getsize(f'model/{f}') / 1024
    print(f'  model/{f}  ({size:.1f} KB)')

In [ ]:
# Download everything as a single zip
from google.colab import files
files.download('model_artifacts.zip')
print('✓ Download started — extract and place pkl files in ml/model/')

## 15 · Summary

| Metric | Result | Target |
|--------|--------|--------|
| ROC-AUC | see above | > 0.90 |
| Recall (bankrupt) | see above | > 0.75 |
| F1 (bankrupt) | see above | > 0.72 |
| CV Std | see above | < 0.05 |

### After downloading
1. Extract `model_artifacts.zip`
2. Copy the 4 `.pkl` files into `bankruptcy-predictor/ml/model/`
3. Run `git add ml/model/*.pkl && git commit -m 'Update model from Colab' && git push`
4. Render auto-deploys with the new model

### Key design decisions
- **Stratified split first** — test set is sealed before any fitting
- **SMOTE on train only** — no synthetic samples leak into test
- **Scaler fit on train only** — no data leakage
- **OOF threshold tuning** — threshold calibrated on real class distribution, not SMOTE-balanced data
- **Scoring = recall** — prioritise catching bankruptcies over precision